# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iemanmalik/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
!pip -q install datasets duckdb huggingface_hub pyarrow pandas

In [16]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("Token loaded successfully!")
print(token[:10] + "...")

Token loaded successfully!
hf_bufmoid...


In [17]:
!pip -q install duckdb datasets huggingface_hub

In [18]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=token
)

print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

For my selected lane (Refresh / Content Opportunity Scoring), one row represents the daily performance of one content page for one client on one report date.

I will use the fact_content_daily_performance table and focus on a mid-panel month (March 2026) to avoid using the final month as my development period.

My goal is to identify content pages that should be reviewed first based on observable search and engagement signals. This work supports decision-making rather than proving that a content update will improve performance.
---



In [19]:
print(type(dataset[0]["report_date"]))
print(dataset[0]["report_date"])

<class 'datetime.date'>
2025-01-27


In [20]:
!pip -q install duckdb huggingface_hub

In [21]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

In [22]:
result = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(result)

   total_rows
0     9841378


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded
## Features, Label, Context and Excluded Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These are observable signals that are available before making a content review decision.

### Label / Proxy
For this assignment I will later define a decline label based on changes in impressions over time.

### Context
- client_hash_id
- content_hash_id
- report_date

These fields identify the client, content item and time period.

### Excluded
I exclude future information and any label-derived columns because they would introduce data leakage. I also exclude any information that could reveal client identity.
*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain = con.sql(f"""
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

grain

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [25]:
# Verification Query 2 (Row count + Date span)

summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

summary

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [26]:
# Verification Query 3 (Availability)

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset is useful for ranking content review opportunities, but it has limitations.

- Different clients have different history lengths (unbalanced panel).
- Some rows contain only Google Search Console data because GA4 tracking started later.
- This analysis is observational and cannot prove that refreshing content causes performance improvements.
- Care must be taken to avoid leakage by ensuring future information is never used as model input.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.